# ML Homework 04: Logistic Regression

## Описание

В этом задании вы реализуете логистическую регрессию для задачи бинарной классификации.
В отличие от линейной регрессии, которая предсказывает вещественное число, логистическая регрессия предсказывает вероятность принадлежности к классу (от 0 до 1).

### Математическое описание

**1. Гипотеза (Hypothesis):**
Модель использует сигмоиду для преобразования линейной комбинации признаков в вероятность:
$$ h_\theta(x) = \sigma(z) = \frac{1}{1 + e^{-z}} $$
где $ z = Xw $ (если смещение $b$ включено в вектор весов $w$ как $w_0$ при $x_0=1$).

**2. Функция потерь (Cost Function):**
Мы используем функцию потерь **Log Loss** (Binary Cross-Entropy), так как MSE здесь не подходит (функция невыпуклая для сигмоиды):
$$ J(w) = -\frac{1}{m} \sum_{i=1}^m [y^{(i)} \log(h_\theta(x^{(i)})) + (1 - y^{(i)}) \log(1 - h_\theta(x^{(i)}))] $$

**3. Градиентный спуск (Gradient Descent):**
Правило обновления весов выглядит так же, как в линейной регрессии, но с другой гипотезой $h_\theta(x)$:
$$ \frac{\partial J}{\partial w} = \frac{1}{m} X^T (h_\theta(X) - y) $$
$$ w := w - \alpha \frac{\partial J}{\partial w} $$
где $\alpha$ — скорость обучения (learning rate).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression as SklearnLogReg
from sklearn.metrics import accuracy_score
import ipytest
import pytest

%matplotlib inline

## Часть 1: Вспомогательные функции

Реализуйте сигмоидную функцию активации.

In [ ]:
def sigmoid(z):
    """
    Вычисляет сигмоиду: 1 / (1 + e^(-z))
    z: скаляр или массив numpy
    """
    # TODO: Реализовать функцию
    return z # Заглушка

## Часть 2: Класс LogisticRegression

In [ ]:
class MyLogisticRegression:
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None

    def fit(self, X, y):
        """
        Обучение модели с помощью градиентного спуска.
        X: матрица признаков (n_samples, n_features)
        y: вектор целевых значений (n_samples,)
        """
        n_samples, n_features = X.shape

        # Инициализация весов нулями
        self.weights = np.zeros(n_features)
        self.bias = 0

        # Градиентный спуск
        for _ in range(self.n_iterations):
            # 1. Линейная модель: z = X * w + b
            # linear_model = ...

            # 2. Предсказание (гипотеза): h = sigmoid(z)
            # y_predicted = ...

            # 3. Вычисление градиентов
            # dw = (1 / n_samples) * (X.T dot (y_predicted - y))
            # db = (1 / n_samples) * sum(y_predicted - y)

            # 4. Обновление параметров
            # self.weights -= ...
            # self.bias -= ...
            pass

    def predict_proba(self, X):
        """
        Предсказание вероятности класса 1.
        Returns: массив вероятностей
        """
        # TODO: Реализовать вычисление вероятностей (sigmoid(Xw + b))
        return np.zeros(X.shape[0])

    def predict(self, X, threshold=0.5):
        """
        Предсказание классов (0 или 1).
        Returns: массив меток классов [0, 1, 1, ...]
        """
        # TODO: Преобразовать вероятности в классы по порогу
        # Если proba > threshold -> 1, иначе 0
        return np.zeros(X.shape[0])

## Часть 3: Визуализация и проверка

In [ ]:
# Генерация данных
X, y = make_classification(n_samples=200, n_features=2, n_redundant=0,
                           n_informative=2, random_state=1, n_clusters_per_class=1)

# Обучение вашей модели
model = MyLogisticRegression(learning_rate=0.1, n_iterations=2000)
model.fit(X, y)
y_pred = model.predict(X)

# Сравнение с Sklearn
sk_model = SklearnLogReg()
sk_model.fit(X, y)
y_pred_sk = sk_model.predict(X)

print(f"Accuracy (My): {accuracy_score(y, y_pred):.4f}")
print(f"Accuracy (Sklearn): {accuracy_score(y, y_pred_sk):.4f}")

# Визуализация границы принятия решений
plt.figure(figsize=(10, 6))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='bwr', edgecolor='k', s=50)

# Рисуем границу (w1*x1 + w2*x2 + b = 0 => x2 = -(w1*x1 + b)/w2)
x1_vals = np.linspace(X[:, 0].min(), X[:, 0].max(), 100)
if model.weights is not None:
    w1, w2 = model.weights
    b = model.bias
    x2_vals = -(w1 * x1_vals + b) / w2
    plt.plot(x1_vals, x2_vals, 'g-', linewidth=2, label='Decision Boundary')

plt.legend()
plt.title('Logistic Regression Decision Boundary')
plt.show()

## Часть 4: Гиперплоскость и проблема XOR

### Что такое гиперплоскость?

В логистической регрессии **гиперплоскость** — это граница принятия решений (decision boundary), которая разделяет пространство признаков на две области.

**Для 2D случая (два признака):**
- Гиперплоскость — это **прямая линия**
- Уравнение: $w_1x_1 + w_2x_2 + b = 0$
- В визуализации выше зелёная линия — это и есть гиперплоскость

**Для 3D случая (три признака):**
- Гиперплоскость — это **плоскость** в 3D пространстве

**Для n признаков:**
- Гиперплоскость — это $(n-1)$-мерная поверхность в $n$-мерном пространстве

**Важное ограничение:** Логистическая регрессия может строить **только линейные** границы!

In [ ]:
# Проблема XOR: когда линейная граница не работает!

# Создаём данные XOR
X_xor = np.array(
    [
        [0, 0],  # класс 0
        [1, 1],  # класс 0
        [0, 1],  # класс 1
        [1, 0],  # класс 1
    ]
)
y_xor = np.array([0, 0, 1, 1])

# Пытаемся обучить логистическую регрессию
model_xor = MyLogisticRegression(learning_rate=0.1, n_iterations=1000)
model_xor.fit(X_xor, y_xor)

# Визуализация
plt.figure(figsize=(12, 5))

# Левый график: данные XOR
plt.subplot(1, 2, 1)
plt.scatter(X_xor[:, 0], X_xor[:, 1], c=y_xor, cmap="bwr", s=200, edgecolors="black", linewidth=2)
plt.title("Данные XOR (нельзя разделить одной прямой!)")
plt.xlabel("x₁")
plt.ylabel("x₂")
plt.grid(True, alpha=0.3)

# Правый график: попытка логистической регрессии
plt.subplot(1, 2, 2)
plt.scatter(X_xor[:, 0], X_xor[:, 1], c=y_xor, cmap="bwr", s=200, edgecolors="black", linewidth=2)

if model_xor.weights is not None:
    w1, w2 = model_xor.weights
    b = model_xor.bias
    x1_line = np.linspace(-0.5, 1.5, 100)
    x2_line = -(w1 * x1_line + b) / w2
    plt.plot(x1_line, x2_line, "g-", linewidth=2, label="Логистическая регрессия")

plt.title("Результат: плохая классификация!")
plt.xlabel("x₁")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Предсказания логистической регрессии на XOR:")
print(f"Точность: {accuracy_score(y_xor, model_xor.predict(X_xor)):.2f}")
print(f"Ожидалось: [0, 0, 1, 1]")
print(f"Получено: {model_xor.predict(X_xor)}")

### Как решать проблему XOR?

Поскольку XOR — это **нелинейно разделимая** задача, линейные модели (логистическая регрессия) здесь бессильны. Вот основные подходы к решению:

#### 1️⃣ **Полиномиальные признаки (Feature Engineering)**
Добавляем новые комбинированные признаки, чтобы сделать данные линейно разделимыми:

```python
# Исходные признаки: [x₁, x₂]
# Новые признаки: [x₁, x₂, x₁x₂, x₁², x₂², ...]

# Например, для XOR достаточно добавить x₁x₂:
# (0,0)→0, (1,1)→1, (0,1)→0, (1,0)→0
# Тогда класс = x₁ + x₂ - 2*x₁x₂
```

#### 2️⃣ **Нейронные сети** 
Используют несколько слоёв с нелинейными активациями для создания сложных границ:
- Входной слой: 2 нейрона (x₁, x₂)
- Скрытый слой: нелинейное преобразование
- Выходной слой: классификация

#### 3️⃣ **Ядерные методы (Kernel Methods)**
Например, SVM с RBF-ядром может нелинейно преобразовать пространство.

#### 4️⃣ **Деревья решений и ансамбли**
Деревья строят piecewise-constant границы, легко справляются с XOR.

#### 5️⃣ **Нейросети с одним скрытым слоем**
Минимальная конфигурация для XOR:
- Вход: 2 нейрона
- Скрытый слой: 2 нейрона (с нелинейной активацией, например ReLU или sigmoid)
- Выход: 1 нейрон

In [ ]:
# Практический пример: решение XOR с помощью полиномиальных признаков

# Добавляем полиномиальные признаки
def add_polynomial_features(X, degree=2):
    """
    Добавляет полиномиальные признаки до указанной степени.
    Для degree=2 добавляет: x₁², x₂², x₁x₂
    """
    X_poly = X.copy()

    # Добавляем x₁x₂ (взаимодействие признаков)
    X_poly = np.column_stack([X_poly, X[:, 0] * X[:, 1]])

    # Добавляем квадраты признаков
    if degree >= 2:
        X_poly = np.column_stack([X_poly, X[:, 0] ** 2, X[:, 1] ** 2])

    return X_poly


# Создаём расширенные признаки для XOR
X_xor_poly = add_polynomial_features(X_xor, degree=2)

print("Исходные признаки:")
print(X_xor)
print("\nПолиномиальные признаки [x₁, x₂, x₁x₂, x₁², x₂²]:")
print(X_xor_poly)

# Обучаем логистическую регрессию на новых признаках
model_poly = MyLogisticRegression(learning_rate=0.1, n_iterations=2000)
model_poly.fit(X_xor_poly, y_xor)

# Проверяем результаты
y_pred_poly = model_poly.predict(X_xor_poly)

print("\nРезультат с полиномиальными признаками:")
print(f"Точность: {accuracy_score(y_xor, y_pred_poly):.2f}")
print(f"Ожидалось: {y_xor}")
print(f"Получено: {y_pred_poly}")

if accuracy_score(y_xor, y_pred_poly) == 1.0:
    print("\n✅ Успех! XOR решён с помощью полиномиальных признаков!")
else:
    print("\n❌ Нужно больше итераций или более сложные признаки")

In [ ]:
ipytest.autoconfig()

class TestLogisticRegression:

    def test_sigmoid(self):
        """Тест функции активации"""
        assert sigmoid(0) == 0.5
        assert np.isclose(sigmoid(100), 1.0)
        assert np.isclose(sigmoid(-100), 0.0)
        assert np.allclose(sigmoid(np.array([0, 0])), np.array([0.5, 0.5]))

    def test_fit_predict(self):
        """Тест обучения на простых разделимых данных"""
        # Создаем простые данные: класс 0 в квадрате [-2, -1], класс 1 в [1, 2]
        X = np.array([[-1, -1], [-2, -2], [1, 1], [2, 2]])
        y = np.array([0, 0, 1, 1])

        model = MyLogisticRegression(learning_rate=0.1, n_iterations=1000)
        model.fit(X, y)

        # Проверяем предсказания
        preds = model.predict(X)
        assert np.array_equal(preds, y)

        # Проверяем вероятности
        probs = model.predict_proba(X)
        assert np.all(probs[:2] < 0.5) # Для класса 0
        assert np.all(probs[2:] > 0.5) # Для класса 1

    def test_comparison_sklearn(self):
        """Сравнение качества с эталонной реализацией"""
        X, y = make_classification(n_samples=100, n_features=4, random_state=42)

        my_model = MyLogisticRegression(learning_rate=0.1, n_iterations=2000)
        my_model.fit(X, y)
        my_acc = accuracy_score(y, my_model.predict(X))

        sk_model = SklearnLogReg()
        sk_model.fit(X, y)
        sk_acc = accuracy_score(y, sk_model.predict(X))

        # Допускаем небольшое отклонение (наши гиперпараметры фиксированы)
        assert my_acc >= sk_acc - 0.1

ipytest.run()